# 09 · Leaching and washing

Sugar from beet, oil from soybeans, copper from ore, caffeine from coffee: leaching dissolves a solute out
of a solid with a solvent, and the solid carries solution with it from stage to stage. With constant underflow
the countercurrent cascade obeys a Kremser-type equation with a washing factor in place of the absorption
factor; with variable underflow the stages are stepped on a diagram. This notebook designs a countercurrent
leaching train, sizes it for a recovery target, and compares it with cross-current washing.

**What you will learn**
- design countercurrent leaching with constant underflow
- size the cascade for a recovery target
- compare cross-current and countercurrent washing
- handle variable underflow stage by stage

**Before you start:** notebook 08  ·  **Time:** about 45 min

In [1]:
try:                      # on Google Colab (or anywhere engsep is missing): install it from GitHub
    import engsep
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engsep"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engsep import extraction as ex
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Countercurrent leaching with constant underflow

In [2]:
solute_in_feed = 100.0                                              # kg of solute per batch of solid
for W in (0.8, 1.0, 1.5, 2.0, 3.0):
    row = [f"{ex.leaching_countercurrent(N, W * 1.0, 1.0, solute_in_feed)['recovery']:.3f}" for N in (1, 2, 4, 8)]
    print(f"washing factor W = {W}: recovery with 1, 2, 4, 8 stages = {row}")

washing factor W = 0.8: recovery with 1, 2, 4, 8 stages = ['0.444', '0.590', '0.703', '0.769']
washing factor W = 1.0: recovery with 1, 2, 4, 8 stages = ['0.500', '0.667', '0.800', '0.889']
washing factor W = 1.5: recovery with 1, 2, 4, 8 stages = ['0.600', '0.789', '0.924', '0.987']
washing factor W = 2.0: recovery with 1, 2, 4, 8 stages = ['0.667', '0.857', '0.968', '0.998']
washing factor W = 3.0: recovery with 1, 2, 4, 8 stages = ['0.750', '0.923', '0.992', '1.000']


The washing factor is the ratio of overflow solution to the solution retained in the underflow: below 1 the
solvent cannot carry the solute away however many stages; above 1 recovery climbs towards complete. The
numbers are the Kremser equation again - leaching, absorption and extraction are one calculation with
different names for the ratio that matters.

## Stages for a recovery target

In [3]:
for W in (1.5, 2.0, 3.0):
    for rec in (0.95, 0.99, 0.999):
        print(f"W = {W}, recovery {rec}: {ex.leaching_stages(rec, W):.1f} stages")

W = 1.5, recovery 0.95: 4.9 stages
W = 1.5, recovery 0.99: 8.7 stages
W = 1.5, recovery 0.999: 14.3 stages
W = 2.0, recovery 0.95: 3.4 stages
W = 2.0, recovery 0.99: 5.7 stages
W = 2.0, recovery 0.999: 9.0 stages
W = 3.0, recovery 0.95: 2.4 stages
W = 3.0, recovery 0.99: 3.8 stages
W = 3.0, recovery 0.999: 5.9 stages


Each extra "nine" of recovery costs about the same number of stages, and more solvent per stage buys fewer
stages: the same trade as in extraction, with the solvent's recovery (evaporation of the extract) as the
cost on the other side. Sugar-beet diffusers run 12-16 stages; oilseed extractors 6-10.

## Cross-current washing: a filter cake

In [4]:
retained = 0.5                                                      # kg solution per kg dry solid in the cake
for n_washes, wash_each in ((1, 2.0), (2, 1.0), (4, 0.5), (8, 0.25)):
    frac_left = 1.0
    for _ in range(n_washes):
        frac_left *= retained / (retained + wash_each)                 # each wash dilutes the retained solution and displaces it
    print(f"{n_washes} washes of {wash_each} kg/kg (2 kg/kg total): {1 - frac_left:.3f} of the solute removed")
print(f"countercurrent with 2 kg/kg in 4 stages: {ex.leaching_countercurrent(4, 2.0, 0.5, 1.0)['recovery']:.3f}")

1 washes of 2.0 kg/kg (2 kg/kg total): 0.800 of the solute removed
2 washes of 1.0 kg/kg (2 kg/kg total): 0.889 of the solute removed
4 washes of 0.5 kg/kg (2 kg/kg total): 0.938 of the solute removed
8 washes of 0.25 kg/kg (2 kg/kg total): 0.961 of the solute removed
countercurrent with 2 kg/kg in 4 stages: 0.997


Washing a cake on a filter is cross-current leaching: fresh wash each time, so splitting the water into
more washes removes more, and countercurrent contact removes more still. The ideal displacement washing of a
real filter does better than this well-mixed model - the wash pushes the mother liquor out as a front - but
the ranking is the same.

## Variable underflow: a stepped construction

In [5]:
# underflow retains solution in proportion to the solution's solute concentration (viscous solutions cling): r = 0.4 + 0.6 c
def underflow_ratio(c):
    return 0.4 + 0.6 * c
def countercurrent_variable(N, solvent, solute_feed, solid=1.0):
    # solve stage by stage with iteration on the overflow concentration profile
    c = np.full(N, 0.5 * solute_feed / (solvent + solute_feed))
    for _ in range(500):
        c_new = c.copy()
        # stage n receives underflow from n-1 (retained r(c_{n-1}) solid at c_{n-1}) and overflow from n+1 (at c_{n+1}); it leaves at c_n both ways
        for n in range(N):
            r_in = solid * underflow_ratio(c[n - 1]) if n > 0 else 0.0
            sol_in_under = solute_feed if n == 0 else r_in * c[n - 1]
            liq_in_under = (solid * underflow_ratio(c[n - 1]) if n > 0 else 0.0)
            over_in = solvent if n == N - 1 else (solvent - 0.0)
            # solute balance: sol_in_under + over_in * c_{n+1} = r(c_n) c_n + over_out c_n  with over_out = liq_in_under + over_in - r(c_n)
            c_next = c[n + 1] if n < N - 1 else 0.0
            def f(cn):
                r_out = solid * underflow_ratio(cn)
                over_out = liq_in_under + over_in - r_out + (0 if n > 0 else 0)
                return sol_in_under + over_in * c_next - r_out * cn - over_out * cn
            lo, hi = 0.0, 1.0
            for _ in range(60):
                mid = 0.5 * (lo + hi)
                if f(mid) > 0:
                    lo = mid
                else:
                    hi = mid
            c_new[n] = mid
        if np.max(np.abs(c_new - c)) < 1e-10:
            break
        c = c_new
    r_last = solid * underflow_ratio(c[-1])
    return 1 - r_last * c[-1] / solute_feed, c
for N in (2, 4, 6):
    rec, prof = countercurrent_variable(N, 3.0, 1.0)
    print(f"{N} stages, 3 kg solvent per kg solid: recovery {rec:.3f}; concentrations {np.round(prof, 3)}")

2 stages, 3 kg solvent per kg solid: recovery 0.968; concentrations [0.405 0.071]
4 stages, 3 kg solvent per kg solid: recovery 0.999; concentrations [0.418 0.085 0.012 0.001]
6 stages, 3 kg solvent per kg solid: recovery 1.000; concentrations [0.418 0.085 0.013 0.002 0.    0.   ]


When the underflow retains more solution at high concentration, the first stages carry more solute in the
solid and the cascade needs a stage-by-stage balance; the library's Kremser functions do not apply, and a
short iteration on the concentration profile does. The result is a lower recovery than the constant-underflow
estimate - the direction real leaching data usually take.

## Inside the algorithm: the washing factor

In [6]:
W_, N_ = 2.0, 4
print("by hand: recovery =", (W_ ** (N_ + 1) - W_) / (W_ ** (N_ + 1) - 1), "; library:", ex.leaching_countercurrent(N_, 2.0, 1.0, 1.0)["recovery"])

by hand: recovery = 0.967741935483871 ; library: 0.967741935483871


## Exercises
1. Sugar beet cossettes (17 % sugar, retaining 1.2 kg solution per kg dry pulp) are leached countercurrently with water to 99 % recovery. Find the stages for water-to-pulp ratios of 1.5, 2 and 3, and the sugar concentration of the juice. Which ratio gives the juice needing least evaporation?
2. Add a solute loss in the underflow moisture after pressing (the pulp is pressed to 0.6 kg/kg after the last stage): how does the recovery change, and how many extra stages compensate?
3. **Implement it yourself:** simulate a 4-stage countercurrent leaching train by iterating the stage balances until the concentrations converge (constant underflow), and confirm the closed-form recovery.